# Preliminary Fisher covariance diagnostics

This notebook diagnoses the covariance used by the baseline-deproj0 Fisher calculation. It addresses two separate questions:

1. **Is the saved covariance numerically stable and scientifically representative of fixed-parameter noise?**
2. **Could the small covariance calculation itself explain a multi-hour runtime?**

The local and random covariance matrices were estimated from 20,000 row-matched noisy-minus-clean residuals. Those simulations vary all nine Battaglia parameters; they are not repeated noise realizations at fixed $\theta$. Strong residual dependence on $\theta$ therefore invalidates the usual fixed-covariance Fisher interpretation even when the matrix inversion is numerically stable.

Only the final $40\times40$ covariance matrices were saved. A true covariance-versus-sample-count convergence test requires the binned residual samples or separately saved covariance estimates for several row counts.


In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

PARAM_NAMES = [
    "P0", "xc", "beta",
    "alpha_m_P0", "alpha_m_xc", "alpha_m_beta",
    "alpha_z_P0", "alpha_z_xc", "alpha_z_beta",
]
PARAM_LABELS = [
    r"$P_0$", r"$x_{\rm c}$", r"$\beta$",
    r"$\alpha_{m,P_0}$", r"$\alpha_{m,x_{\rm c}}$", r"$\alpha_{m,\beta}$",
    r"$\alpha_{z,P_0}$", r"$\alpha_{z,x_{\rm c}}$", r"$\alpha_{z,\beta}$",
]

# Set this explicitly if Jupyter was launched from an unrelated directory.
ANALYSIS_DIR_OVERRIDE = None
SAVE_FIGURES = False

candidates = [
    Path.cwd(),
    Path.cwd() / "adrian_fisher_baseline_deproj0" / "analysis",
    Path.cwd() / "SBI_analysis" / "adrian_fisher_baseline_deproj0" / "analysis",
    Path("/home/cbllover/HalfDome/SBI_analysis/adrian_fisher_baseline_deproj0/analysis"),
]
if ANALYSIS_DIR_OVERRIDE is not None:
    candidates.insert(0, Path(ANALYSIS_DIR_OVERRIDE))

ANALYSIS_DIR = next(
    (path.resolve() for path in candidates if (path / "covariance_shrunk.npy").is_file()),
    None,
)
if ANALYSIS_DIR is None:
    raise FileNotFoundError(
        "Could not find covariance_shrunk.npy. Set ANALYSIS_DIR_OVERRIDE in this cell."
    )

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

def finish_figure(fig, filename):
    fig.tight_layout()
    if SAVE_FIGURES:
        path = ANALYSIS_DIR / filename
        fig.savefig(path)
        print("Saved:", path)
    plt.show()

print("Analysis directory:", ANALYSIS_DIR)


## 1. Inventory and pipeline stage

The stage table is useful for runtime diagnosis. If the covariance, inverse, and Fisher samples exist but the SBI sample or corner plots do not, the run passed covariance construction and stalled later.


In [ ]:
expected_stages = {
    "covariance matrices": ["covariance_local_sample.npy", "covariance_random_sample.npy", "covariance_shrunk.npy"],
    "inverse covariance": ["inverse_covariance.npy"],
    "Fisher matrix": ["fisher_matrix_normalized.npy", "fisher_covariance_normalized.npy"],
    "Fisher samples": ["fisher_samples_untruncated.npy", "fisher_samples_prior_truncated.npy"],
    "saved NPE samples": list(ANALYSIS_DIR.glob("sbi_samples_N*.npy")),
    "corner plots": list(ANALYSIS_DIR.glob("fisher_*_corner.jpg")),
    "final summary": ["fisher_analysis_summary.json"],
}

stage_rows = []
for stage, items in expected_stages.items():
    if items and isinstance(items[0], Path):
        paths = items
    else:
        paths = [ANALYSIS_DIR / item for item in items]
    stage_rows.append({
        "stage": stage,
        "complete": bool(paths) and all(path.is_file() for path in paths),
        "files_found": sum(path.is_file() for path in paths),
        "files_expected": len(paths),
    })
display(pd.DataFrame(stage_rows))

inventory = []
for path in sorted(ANALYSIS_DIR.iterdir()):
    if path.is_file():
        inventory.append({
            "file": path.name,
            "size_MiB": path.stat().st_size / 2**20,
        })
display(pd.DataFrame(inventory).sort_values("file").reset_index(drop=True))


## 2. Load and validate the saved matrices

Condition numbers are shown for both covariance and correlation matrices. The correlation condition number is scale-independent and is generally the more useful diagnostic here.


In [ ]:
C_local = np.load(ANALYSIS_DIR / "covariance_local_sample.npy")
C_random = np.load(ANALYSIS_DIR / "covariance_random_sample.npy")
C_shrunk = np.load(ANALYSIS_DIR / "covariance_shrunk.npy")
precision_saved = np.load(ANALYSIS_DIR / "inverse_covariance.npy")
local_indices = np.load(ANALYSIS_DIR / "covariance_local_indices.npy")
random_indices = np.load(ANALYSIS_DIR / "covariance_random_indices.npy")
residual_r2 = np.load(ANALYSIS_DIR / "residual_r_squared.npy")
residual_param_corr = np.load(ANALYSIS_DIR / "residual_parameter_correlation.npy")
derivatives = np.load(ANALYSIS_DIR / "derivatives_richardson.npy")
fisher_q = np.load(ANALYSIS_DIR / "fisher_matrix_normalized.npy")
fisher_cov_q = np.load(ANALYSIS_DIR / "fisher_covariance_normalized.npy")
fisher_cov_theta = np.load(ANALYSIS_DIR / "fisher_covariance_theta.npy")

matrices = {
    "local sample": C_local,
    "random sample": C_random,
    "5% diagonal shrinkage": C_shrunk,
}

def covariance_to_correlation(matrix):
    sigma = np.sqrt(np.clip(np.diag(matrix), 0.0, None))
    return np.divide(
        matrix,
        np.outer(sigma, sigma),
        out=np.zeros_like(matrix, dtype=float),
        where=np.outer(sigma, sigma) > 0.0,
    )

def effective_ranks(eigenvalues):
    eigenvalues = np.clip(np.asarray(eigenvalues, dtype=float), 0.0, None)
    if eigenvalues.sum() == 0.0:
        return np.nan, np.nan
    participation = eigenvalues.sum() ** 2 / np.sum(eigenvalues**2)
    probabilities = eigenvalues[eigenvalues > 0.0] / eigenvalues.sum()
    entropy_rank = np.exp(-np.sum(probabilities * np.log(probabilities)))
    return participation, entropy_rank

def matrix_summary(name, matrix):
    symmetric = 0.5 * (matrix + matrix.T)
    correlation = covariance_to_correlation(symmetric)
    eigenvalues = np.linalg.eigvalsh(symmetric)
    corr_eigenvalues = np.linalg.eigvalsh(0.5 * (correlation + correlation.T))
    participation, entropy_rank = effective_ranks(corr_eigenvalues)
    off_diagonal = correlation - np.eye(correlation.shape[0])
    return {
        "matrix": name,
        "shape": str(matrix.shape),
        "finite": bool(np.all(np.isfinite(matrix))),
        "symmetry_error": np.linalg.norm(matrix - matrix.T) / np.linalg.norm(matrix),
        "min_variance": np.diag(matrix).min(),
        "min_eigenvalue": eigenvalues.min(),
        "covariance_condition": np.linalg.cond(symmetric),
        "correlation_condition": np.linalg.cond(correlation),
        "correlation_participation_rank": participation,
        "correlation_entropy_rank": entropy_rank,
        "mean_abs_offdiag_correlation": np.sum(np.abs(off_diagonal)) / (matrix.shape[0] * (matrix.shape[0] - 1)),
    }

matrix_table = pd.DataFrame(
    [matrix_summary(name, matrix) for name, matrix in matrices.items()]
)
display(matrix_table)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18.0 / 2.54, 6.2 / 2.54), sharex=True, sharey=True)
image = None
for axis, (name, matrix) in zip(axes, matrices.items()):
    correlation = covariance_to_correlation(matrix)
    image = axis.imshow(correlation, origin="lower", vmin=-1, vmax=1, cmap="RdBu_r")
    axis.set_title(name)
    axis.set_xlabel("$D_\ell$ bin")
axes[0].set_ylabel("$D_\ell$ bin")
fig.colorbar(image, ax=axes, fraction=0.025, pad=0.02, label="Correlation")
finish_figure(fig, "preliminary_covariance_correlation_matrices.jpg")


## 3. Eigenmodes and effective dimensionality

A healthy 40-bin covariance need not have 40 equally informative modes, but a correlation matrix dominated by one or two modes is vulnerable to modeling errors. Diagonal shrinkage improves invertibility without proving that the covariance is scientifically appropriate.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18.0 / 2.54, 6.5 / 2.54))
for name, matrix in matrices.items():
    correlation = covariance_to_correlation(matrix)
    eigenvalues = np.linalg.eigvalsh(0.5 * (correlation + correlation.T))[::-1]
    axes[0].semilogy(np.arange(1, len(eigenvalues) + 1), eigenvalues, marker="o", ms=2.5, lw=0.9, label=name)
    axes[1].plot(np.arange(1, len(eigenvalues) + 1), np.cumsum(eigenvalues) / eigenvalues.sum(), lw=1.0, label=name)

axes[0].set(xlabel="Ordered mode", ylabel="Correlation eigenvalue")
axes[1].set(xlabel="Number of modes", ylabel="Cumulative correlation variance", ylim=(0, 1.02))
for axis in axes:
    axis.grid(True, alpha=0.25)
    axis.legend(frameon=False)
finish_figure(fig, "preliminary_covariance_eigenmodes.jpg")


## 4. Local versus random covariance

For 20,000 independent Gaussian residuals, the approximate fractional sampling uncertainty of a variance is $\sqrt{2/(N-1)}\simeq1\%$. Differences of tens of percent therefore cannot plausibly be explained by finite covariance-sample noise alone.


In [ ]:
n_covariance = len(local_indices)
n_bins = C_local.shape[0]
sigma_local = np.sqrt(np.diag(C_local))
sigma_random = np.sqrt(np.diag(C_random))
sigma_ratio = sigma_local / sigma_random
covariance_frobenius_difference = (
    np.linalg.norm(C_local - C_random, ord="fro")
    / np.linalg.norm(C_random, ord="fro")
)
expected_variance_fractional_error = np.sqrt(2.0 / (n_covariance - 1.0))

comparison = pd.Series({
    "N covariance rows": n_covariance,
    "number of bins": n_bins,
    "expected 1-sigma fractional variance error": expected_variance_fractional_error,
    "local/random covariance Frobenius difference": covariance_frobenius_difference,
    "minimum local/random sigma ratio": sigma_ratio.min(),
    "median local/random sigma ratio": np.median(sigma_ratio),
    "maximum local/random sigma ratio": sigma_ratio.max(),
}, name="value")
display(comparison.to_frame())

bins = np.arange(n_bins)
fig, axes = plt.subplots(1, 2, figsize=(18.0 / 2.54, 6.2 / 2.54))
axes[0].plot(bins, sigma_ratio, marker="o", ms=2.5, lw=0.9)
axes[0].axhline(1.0, color="black", lw=0.8)
axes[0].axhspan(0.8, 1.25, color="0.85", label="0.8--1.25 diagnostic band")
axes[0].set(xlabel="$D_\ell$ bin", ylabel=r"$\sigma_{\rm local}/\sigma_{\rm random}$")
axes[0].legend(frameon=False)

axes[1].semilogy(bins, sigma_local, marker="o", ms=2.5, lw=0.9, label="local")
axes[1].semilogy(bins, sigma_random, marker="s", ms=2.5, lw=0.9, label="random")
axes[1].set(xlabel="$D_\ell$ bin", ylabel="Residual standard deviation")
axes[1].legend(frameon=False)
for axis in axes:
    axis.grid(True, alpha=0.25)
finish_figure(fig, "preliminary_local_random_covariance.jpg")


## 5. Does the noisy-minus-clean residual depend on the parameters?

The code's own warning threshold is $R^2>0.05$. If many bins exceed it, the residual ensemble is not behaving like parameter-independent noise. In that case the local covariance is an approximation to $C(\theta)$ over a finite neighborhood, not a fixed-$\theta$ covariance.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18.0 / 2.54, 10.0 / 2.54), gridspec_kw={"height_ratios": [1, 2]})
axes[0].plot(np.arange(n_bins), residual_r2, marker="o", ms=2.5, lw=0.9)
axes[0].axhline(0.05, color="black", ls="--", lw=0.8, label="$R^2=0.05$ warning threshold")
axes[0].set(xlabel="$D_\ell$ bin", ylabel=r"Residual parameter $R^2$")
axes[0].legend(frameon=False)
axes[0].grid(True, alpha=0.25)

corr_image = axes[1].imshow(residual_param_corr, aspect="auto", origin="lower", vmin=-1, vmax=1, cmap="RdBu_r")
axes[1].set_yticks(np.arange(len(PARAM_LABELS)))
axes[1].set_yticklabels(PARAM_LABELS)
axes[1].set(xlabel="$D_\ell$ bin", ylabel="Parameter")
fig.colorbar(corr_image, ax=axes[1], fraction=0.025, pad=0.02, label="Residual--parameter correlation")
finish_figure(fig, "preliminary_residual_parameter_dependence.jpg")

print("R^2 range:", residual_r2.min(), "to", residual_r2.max())
print("Bins above R^2=0.05:", np.count_nonzero(residual_r2 > 0.05), "/", n_bins)
print("Maximum absolute residual--parameter correlation:", np.max(np.abs(residual_param_corr)))


## 6. Precision-matrix consistency

The saved precision matrix includes the Hartlap factor. Therefore $C\,\widehat C^{-1}$ should equal the Hartlap factor times the identity, rather than exactly the identity.


In [ ]:
hartlap_factor = (n_covariance - n_bins - 2.0) / (n_covariance - 1.0)
precision_product = C_shrunk @ precision_saved
precision_target = hartlap_factor * np.eye(n_bins)
precision_relative_error = (
    np.linalg.norm(precision_product - precision_target, ord="fro")
    / np.linalg.norm(precision_target, ord="fro")
)

precision_report = pd.Series({
    "Hartlap factor": hartlap_factor,
    "relative ||C P - hI||_F": precision_relative_error,
    "maximum absolute off-diagonal of C P": np.max(np.abs(precision_product - np.diag(np.diag(precision_product)))),
    "mean diagonal of C P": np.mean(np.diag(precision_product)),
    "local covariance condition number": np.linalg.cond(C_local),
    "shrunk covariance condition number": np.linalg.cond(C_shrunk),
    "normalized Fisher condition number": np.linalg.cond(fisher_q),
}, name="value")
display(precision_report.to_frame())


## 7. Sensitivity to diagonal shrinkage

This recomputes the Fisher covariance over a shrinkage grid. Large movement of marginalized errors with shrinkage means covariance regularization, rather than derivative information, is controlling the result.


In [ ]:
def regularized_inverse(matrix, eigenvalue_floor=1.0e-10):
    symmetric = 0.5 * (matrix + matrix.T)
    eigenvalues, eigenvectors = np.linalg.eigh(symmetric)
    floor = max(eigenvalues.max() * eigenvalue_floor, np.finfo(float).tiny)
    clipped = np.maximum(eigenvalues, floor)
    inverse = (eigenvectors * (1.0 / clipped)) @ eigenvectors.T
    repaired = (eigenvectors * clipped) @ eigenvectors.T
    return inverse, repaired, int(np.count_nonzero(eigenvalues < floor))

# C_theta = diag(prior_width) C_q diag(prior_width), so the widths can be
# recovered from the saved diagonal elements without loading the dataset.
prior_width = np.sqrt(np.diag(fisher_cov_theta) / np.diag(fisher_cov_q))
derivatives_q = derivatives * prior_width[:, None]
shrinkage_grid = np.array([0.0, 0.01, 0.03, 0.05, 0.10, 0.20, 0.50, 1.0])
sensitivity_rows = []

for shrinkage in shrinkage_grid:
    covariance_trial = (
        (1.0 - shrinkage) * C_local
        + shrinkage * np.diag(np.diag(C_local))
    )
    precision_trial, covariance_repaired, covariance_floored = regularized_inverse(covariance_trial)
    precision_trial *= hartlap_factor
    fisher_trial = derivatives_q @ precision_trial @ derivatives_q.T
    fisher_covariance_trial, fisher_repaired, fisher_floored = regularized_inverse(fisher_trial)
    normalized_std = np.sqrt(np.diag(fisher_covariance_trial))
    for param, std in zip(PARAM_NAMES, normalized_std):
        sensitivity_rows.append({
            "shrinkage": shrinkage,
            "param": param,
            "normalized_std": std,
            "covariance_condition": np.linalg.cond(covariance_repaired),
            "covariance_modes_floored": covariance_floored,
            "fisher_condition": np.linalg.cond(fisher_repaired),
            "fisher_modes_floored": fisher_floored,
        })

sensitivity = pd.DataFrame(sensitivity_rows)
reference = sensitivity[sensitivity["shrinkage"] == 0.05].set_index("param")["normalized_std"]
sensitivity["std_relative_to_5pct"] = sensitivity.apply(
    lambda row: row["normalized_std"] / reference.loc[row["param"]], axis=1
)

fig, ax = plt.subplots(figsize=(10.0 / 2.54, 7.0 / 2.54))
for param, label in zip(PARAM_NAMES, PARAM_LABELS):
    sub = sensitivity[sensitivity["param"] == param]
    ax.plot(sub["shrinkage"], sub["std_relative_to_5pct"], marker="o", ms=2.5, lw=0.8, label=label)
ax.axhline(1.0, color="black", lw=0.8, ls="--")
ax.set(xlabel="Diagonal shrinkage fraction", ylabel="Marginalized std / value at 5% shrinkage")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, ncol=3, fontsize=7)
finish_figure(fig, "preliminary_fisher_shrinkage_sensitivity.jpg")

display(sensitivity.groupby("shrinkage").first()[[
    "covariance_condition", "covariance_modes_floored", "fisher_condition", "fisher_modes_floored"
]])


## 8. Runtime scale

Eigendecomposition and inversion of a $40\times40$ matrix should take milliseconds, not hours. The covariance stage's heavier operation is reading and binning noisy and clean spectra for 20,000 local plus 20,000 random rows. Its minimum float32 input traffic is about 2.5 GB before temporary float64 copies. The saved matrices alone cannot benchmark that original memmap I/O.


In [ ]:
def median_runtime(function, repeats=100):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        function()
        times.append(time.perf_counter() - start)
    return 1.0e3 * np.median(times)

runtime_table = pd.DataFrame([
    {"operation": "40x40 eigendecomposition", "median_ms": median_runtime(lambda: np.linalg.eigh(C_shrunk))},
    {"operation": "40x40 inverse", "median_ms": median_runtime(lambda: np.linalg.inv(C_shrunk))},
    {"operation": "9x40 Fisher multiplication", "median_ms": median_runtime(lambda: derivatives @ precision_saved @ derivatives.T)},
])
display(runtime_table)

n_raw_ell = 7900
minimum_io_gb = (
    (len(local_indices) + len(random_indices))
    * n_raw_ell
    * 2  # noisy and clean
    * np.dtype(np.float32).itemsize
    / 1.0e9
)
print(f"Minimum covariance-stage spectrum input traffic: {minimum_io_gb:.2f} GB")
print("The 40x40 linear algebra is not a plausible multi-hour bottleneck.")


## 9. Automatic assessment

Recommended interpretation:

- **Inverse check passes:** the stored inverse is numerically consistent with the stored shrunk covariance and Hartlap correction.
- **Conditioning warning:** strong mode hierarchy means results can depend on shrinkage and eigenvalue flooring.
- **Scientific covariance warning:** local/random disagreement and residual dependence on the nine parameters show that noisy-minus-clean residuals are not parameter-independent fixed-$\theta$ noise.
- **Runtime:** the covariance's $40\times40$ matrix operations are not the multi-hour bottleneck. Use the stage inventory and PBS log timestamps to distinguish raw memmap binning, truncated sampling, saved-NPE sampling, and GetDist plotting.

For a defensible covariance convergence test, update the cluster analysis to save `local_residuals` and `random_residuals` (each only about 6.4 MB as float64 for 20,000 by 40), then recompute covariance for nested row counts such as 1,000, 2,000, 5,000, 10,000, and 20,000. The preferred solution remains repeated noise realizations at fixed Battaglia12 parameters. If these cannot be generated, explicitly model $C(\theta)$ or state that the local residual covariance is an approximation.


In [ ]:
checks = [
    {
        "diagnostic": "saved inverse consistency",
        "value": precision_relative_error,
        "threshold": "< 1e-8",
        "status": "PASS" if precision_relative_error < 1.0e-8 else "FAIL",
    },
    {
        "diagnostic": "local/random covariance difference",
        "value": covariance_frobenius_difference,
        "threshold": "< 0.1",
        "status": "PASS" if covariance_frobenius_difference < 0.1 else "FAIL",
    },
    {
        "diagnostic": "local/random sigma range",
        "value": f"{sigma_ratio.min():.3f} .. {sigma_ratio.max():.3f}",
        "threshold": "inside 0.8 .. 1.25",
        "status": "PASS" if np.all((sigma_ratio >= 0.8) & (sigma_ratio <= 1.25)) else "FAIL",
    },
    {
        "diagnostic": "maximum residual parameter R2",
        "value": residual_r2.max(),
        "threshold": "< 0.05",
        "status": "PASS" if residual_r2.max() < 0.05 else "FAIL",
    },
    {
        "diagnostic": "shrunk covariance condition",
        "value": np.linalg.cond(C_shrunk),
        "threshold": "inspect if > 1e4",
        "status": "PASS" if np.linalg.cond(C_shrunk) < 1.0e4 else "WARN",
    },
    {
        "diagnostic": "normalized Fisher condition",
        "value": np.linalg.cond(fisher_q),
        "threshold": "inspect if > 1e8",
        "status": "PASS" if np.linalg.cond(fisher_q) < 1.0e8 else "WARN",
    },
]
assessment = pd.DataFrame(checks)
display(assessment)

if (ANALYSIS_DIR / "covariance_diagnostics.jpg").is_file():
    display(Image(filename=str(ANALYSIS_DIR / "covariance_diagnostics.jpg")))
